In [1]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target

print(df.shape)
df.head()

(150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [2]:
df.isnull().sum()

sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
dtype: int64

In [3]:
df.duplicated().sum()

np.int64(1)

In [4]:
print(df['target'].value_counts())

target
0    50
1    50
2    50
Name: count, dtype: int64


In [5]:
df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

def metrics(X_test, y_test, model):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    result = {
        "Accuracy_score": round(accuracy_score(y_test, y_pred), 2),
        "Pecision_score": round(precision_score(y_test, y_pred, average='weighted'), 2),
        "Recall_score": round(recall_score(y_test, y_pred, average='weighted'), 2),
        "F1-score": round(f1_score(y_test, y_pred, average='weighted'), 2),
        "ROC-AUC-score": round(roc_auc_score(y_test, y_proba, multi_class='ovr'), 2)
    }
    return pd.DataFrame(result, index=[0])

In [13]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
gb.fit(X_train, y_train)

metrics(X_test, y_test, gb)

,Accuracy_score,Pecision_score,Recall_score,F1-score,ROC-AUC-score
0,0.97,0.97,0.97,0.97,0.99


In [16]:
# Classification report
y_pred = gb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=iris.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [17]:
# Tajribalar: n_estimators, learning_rate va max_depth ni solishtirish
res = []

for n in [50, 100, 200]:
    for lr in [0.01, 0.1, 0.2]:
        for depth in [1, 2, 3]:
            model = GradientBoostingClassifier(
                n_estimators=n,
                learning_rate=lr,
                max_depth=depth,
                random_state=42
            )
            model.fit(X_train, y_train)
            
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)
            
            res.append({
                "n_estimators": n,
                "learning_rate": lr,
                "max_depth": depth,
                "Accuracy_score": round(accuracy_score(y_test, y_pred), 2),
                "F1-score": round(f1_score(y_test, y_pred, average='weighted'), 2),
                "ROC-AUC-score": round(roc_auc_score(y_test, y_proba, multi_class='ovr'), 2)
            })

results_df = pd.DataFrame(res)
results_df.sort_values(by="Accuracy_score", ascending=False).head(10)

,n_estimators,learning_rate,max_depth,Accuracy_score,F1-score,ROC-AUC-score
1,50,0.01,2,0.97,0.97,0.98
2,50,0.01,3,0.97,0.97,0.96
3,50,0.10,1,0.97,0.97,0.96
10,100,0.01,2,0.97,0.97,0.97
5,50,0.10,3,0.97,0.97,0.99
9,100,0.01,1,0.97,0.97,0.98
8,50,0.20,3,0.97,0.97,0.99
18,200,0.01,1,0.97,0.97,0.96
17,100,0.20,3,0.97,0.97,0.99
14,100,0.10,3,0.97,0.97,0.99
